# Akan-BPE Model Ladder - Light Split

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/run-full-light.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/run-full-light.ipynb)

Runs the lighter half of the ladder: Qwen3-0.6B, Llama-3.2-1B, and tiny-aya-base.

Run this top-to-bottom on Kaggle or Colab with a T4 GPU. Each model runs two arms: random embedding init and mean-of-subword embedding init. After each model, the notebook prints fertility/tokenization metrics, BPB metrics, generation samples, reload verification, the full JSON payloads, and then zips artifacts into `outputs/`.

## Setup

In [1]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Cloning into 'akan-bpe'...
remote: Enumerating objects: 679, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 679 (delta 103), reused 115 (delta 67), pack-reused 516 (from 1)
Receiving objects: 100% (679/679), 1.16 MiB | 9.35 MiB/s, done.
Resolving deltas: 100% (427/427), done.
/kaggle/working/akan-bpe
Working directory: /kaggle/working/akan-bpe


In [2]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 72.0 MB/s eta 0:00:00:00:0100:01


In [3]:
# Hugging Face authentication. Paste a token with access to gated models
# when prompted. The input is hidden.
from huggingface_hub import login

login()

In [4]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a T4 GPU accelerator, then re-run.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [5]:
# Download Akan datasets. --tts-limit 50000 pins the 40,000/5,000/5,000 split.
!python scripts/download.py --output-dir data --tts-limit 50000

README.md: 29.2kB [00:00, 13.2MB/s]
Resolving data files: 100%|████████████████| 270/270 [00:00<00:00, 29182.65it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 14.5MB/s]
Wrote 40000 rows to data/pristine_twi_train.jsonl
Wrote 5000 rows to data/pristine_twi_validation.jsonl
Wrote 5000 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [6]:
# Verify required inputs exist. The TTS tokenizer is committed under models/.
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

All inputs ready.


## Reporting helpers

In [7]:
import json
from pathlib import Path


def load_result(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing result JSON: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def fmt(value, digits=4):
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def result_paths(model_slug):
    return {
        "random": Path("results") / f"run-{model_slug}.json",
        "mean_subword": Path("results") / f"run-{model_slug}-meansub.json",
    }


def print_tokenization_table(results):
    print("\nTOKENIZATION / FERTILITY")
    print("tokens/word is fertility; lower is better.")
    header = (
        f"{'arm':<22}{'base fertility':>16}{'Akan fertility':>16}"
        f"{'base tokens':>14}{'Akan tokens':>14}{'words':>10}{'reduction':>12}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        counts = payload["token_count_comparison"]
        base = counts["base_model_tokenizer"]
        akan = counts["experiment_tokenizer"]
        print(
            f"{arm:<22}"
            f"{fmt(base.get('fertility')):>16}"
            f"{fmt(akan.get('fertility')):>16}"
            f"{base.get('total_tokens'):>14}"
            f"{akan.get('total_tokens'):>14}"
            f"{akan.get('total_words'):>10}"
            f"{fmt(counts.get('token_reduction_ratio')):>12}"
        )


def print_bpb_table(results):
    print("\nMODELING / BPB")
    print("BPB uses full byte coverage; lower experiment BPB is better.")
    header = (
        f"{'arm':<22}{'eval_loss':>12}{'perplexity':>12}"
        f"{'experiment BPB':>18}{'base BPB':>12}{'improvement':>14}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        bpb = payload["eval"]["bpb"]
        base_bpb = (bpb.get("base") or {}).get("bits_per_byte")
        print(
            f"{arm:<22}"
            f"{fmt(payload['eval'].get('eval_loss')):>12}"
            f"{fmt(payload['eval'].get('perplexity'), 2):>12}"
            f"{fmt(bpb['experiment'].get('bits_per_byte')):>18}"
            f"{fmt(base_bpb):>12}"
            f"{fmt(bpb.get('improvement')):>14}"
        )


def print_generation_samples(results):
    print("\nGENERATION SAMPLES")
    for arm, payload in results.items():
        print(f"\n[{arm}]")
        for index, sample in enumerate(payload.get("generation_samples", []), start=1):
            print(f"sample {index} prompt: {sample.get('prompt', '')}")
            print(f"sample {index} completion: {sample.get('completion', '')}")


def print_run_integrity(results):
    print("\nRUN INTEGRITY")
    header = f"{'arm':<22}{'output_model_dir':<34}{'reload ok':>10}{'device':>18}"
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        reload_info = payload.get("reload_verification") or {}
        device = payload.get("device") or {}
        print(
            f"{arm:<22}"
            f"{str(payload.get('output_model_dir')):<34}"
            f"{str(reload_info.get('success')):>10}"
            f"{str(device.get('device_name')):>18}"
        )


def print_full_json(results):
    for arm, payload in results.items():
        print(f"\nBEGIN_FULL_JSON {arm}")
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f"END_FULL_JSON {arm}")


def display_model_results(model_slug):
    paths = result_paths(model_slug)
    results = {arm: load_result(path) for arm, path in paths.items()}
    print("=" * 88)
    print(f"FULL RESULTS FOR {model_slug}")
    print("=" * 88)
    print("Result files:")
    for arm, path in paths.items():
        print(f"- {arm}: {path}")
    print_tokenization_table(results)
    print_bpb_table(results)
    print_generation_samples(results)
    print_run_integrity(results)
    best = min(results, key=lambda arm: results[arm]["eval"]["bpb"]["experiment"]["bits_per_byte"])
    print(f"\nBest downstream BPB arm: {best}")
    print_full_json(results)
    return results


## Run qwen-0.6b

Model id: `Qwen/Qwen3-0.6B-Base`. This section runs both arms, then immediately displays and archives the full results.

In [8]:
# qwen-0.6b / Arm A - random embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-0.6B-Base

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.12MB/s]
tokenizer_config.json: 9.68kB [00:00, 21.5MB/s]
vocab.json: 2.78MB [00:00, 11.6MB/s]
merges.txt: 1.67MB [00:00, 13.0MB/s]
tokenizer.json: 7.03MB [00:00, 21.1MB/s]
Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1831.82text/s]
model.safetensors: 100%|████████████████████| 1.19G/1.19G [00:04<00:00, 257MB/s]
Loading weights: 100%|█| 310/310 [00:00<00:00, 492.47it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 138/138 [00:00<00:00, 1.03MB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(m

In [9]:
# qwen-0.6b / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-0.6B-Base --embedding-init-mode mean_subword

Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1662.13text/s]
Loading weights: 100%|█| 310/310 [00:00<00:00, 524.50it/s, Materializing param=m
Initializing mean-subword embeddings: 100%|█| 8000/8000 [00:01<00:00, 4176.51tok
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '6.711', 'grad_norm': '3.9', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.587', 'grad_norm': '2.095', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.305', 'grad_norm': '2.395', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.066', 'grad_norm': '2.957', 'learning_rate'

## Full results and artifact zip - qwen-0.6b

In [10]:
results_qwen_0_6b = display_model_results("qwen-0.6b")

FULL RESULTS FOR qwen-0.6b
Result files:
- random: results/run-qwen-0.6b.json
- mean_subword: results/run-qwen-0.6b-meansub.json

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
random                          2.5303          1.2586        357830        177986    141420      0.5026
mean_subword                    2.5303          1.2586        357830        177986    141420      0.5026

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is better.
arm                      eval_loss  perplexity    experiment BPB    base BPB   improvement
------------------------------------------------------------------------------------------
random                      4.4150       82.68            1.4805      2.9523        1.4718
mean_subword                3.81

In [11]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-qwen-0.6b-artifacts.zip     results/run-qwen-0.6b.json     results/run-qwen-0.6b-meansub.json     models/run-qwen-0.6b     models/run-qwen-0.6b-meansub
!ls -lh outputs

models:
total 1.8M
-rw-r--r-- 1 root root 522K Jun 13 22:48 asr_tokenizer.json
-rw-r--r-- 1 root root 1.3K Jun 13 22:48 asr_tokenizer_stats.json
-rw-r--r-- 1 root root 515K Jun 13 22:48 mixed_tokenizer.json
-rw-r--r-- 1 root root  12K Jun 13 22:48 mixed_tokenizer_stats.json
-rw-r--r-- 1 root root 227K Jun 13 22:48 router_classifier.pkl
drwxr-xr-x 3 root root 4.0K Jun 13 23:44 run-qwen-0.6b
drwxr-xr-x 3 root root 4.0K Jun 14 00:38 run-qwen-0.6b-meansub
-rw-r--r-- 1 root root 515K Jun 13 22:48 tts_tokenizer.json
-rw-r--r-- 1 root root  11K Jun 13 22:48 tts_tokenizer_stats.json

outputs:
total 0

results:
total 8.0K
-rw-r--r-- 1 root root 3.4K Jun 13 23:44 run-qwen-0.6b.json
-rw-r--r-- 1 root root 3.4K Jun 14 00:38 run-qwen-0.6b-meansub.json
  adding: results/run-qwen-0.6b.json (deflated 66%)
  adding: results/run-qwen-0.6b-meansub.json (deflated 64%)
  adding: models/run-qwen-0.6b/ (stored 0%)
  adding: models/run-qwen-0.6b/README.md (deflated 65%)
  adding: models/run-qwen-0.6b/adapter_

## Run llama-1b

Model id: `meta-llama/Llama-3.2-1B`. This section runs both arms, then immediately displays and archives the full results.

In [12]:
# llama-1b / Arm A - random embedding init
!python scripts/model_integration.py --model-id meta-llama/Llama-3.2-1B

config.json: 100%|█████████████████████████████| 843/843 [00:00<00:00, 3.96MB/s]
tokenizer_config.json: 50.5kB [00:00, 101MB/s]
tokenizer.json: 9.09MB [00:00, 42.4MB/s]
Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1661.77text/s]
model.safetensors: 100%|████████████████████| 2.47G/2.47G [00:15<00:00, 160MB/s]
Loading weights: 100%|█| 146/146 [00:00<00:00, 171.53it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 185/185 [00:00<00:00, 1.04MB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '7.129', 'grad_norm': '2.93', 'learning_rate': '0.000193', 'ep

In [13]:
# llama-1b / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id meta-llama/Llama-3.2-1B --embedding-init-mode mean_subword

Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1635.06text/s]
Loading weights: 100%|█| 146/146 [00:00<00:00, 171.28it/s, Materializing param=m
Initializing mean-subword embeddings: 100%|█| 8000/8000 [00:01<00:00, 4474.70tok
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '7.179', 'grad_norm': '4.935', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.726', 'grad_norm': '2.292', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.353', 'grad_norm': '2.544', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.03', 'grad_norm': '3.555', 'learning_rate

## Full results and artifact zip - llama-1b

In [14]:
results_llama_1b = display_model_results("llama-1b")

FULL RESULTS FOR llama-1b
Result files:
- random: results/run-llama-1b.json
- mean_subword: results/run-llama-1b-meansub.json

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
random                          3.0732          1.2586        434611        177986    141420      0.5905
mean_subword                    3.0732          1.2586        434611        177986    141420      0.5905

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is better.
arm                      eval_loss  perplexity    experiment BPB    base BPB   improvement
------------------------------------------------------------------------------------------
random                      4.2354       69.09            1.4150      2.4480        1.0330
mean_subword                3.6698 

In [15]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-llama-1b-artifacts.zip     results/run-llama-1b.json     results/run-llama-1b-meansub.json     models/run-llama-1b     models/run-llama-1b-meansub
!ls -lh outputs

models:
total 1.8M
-rw-r--r-- 1 root root 522K Jun 13 22:48 asr_tokenizer.json
-rw-r--r-- 1 root root 1.3K Jun 13 22:48 asr_tokenizer_stats.json
-rw-r--r-- 1 root root 515K Jun 13 22:48 mixed_tokenizer.json
-rw-r--r-- 1 root root  12K Jun 13 22:48 mixed_tokenizer_stats.json
-rw-r--r-- 1 root root 227K Jun 13 22:48 router_classifier.pkl
drwxr-xr-x 3 root root 4.0K Jun 14 01:12 run-llama-1b
drwxr-xr-x 3 root root 4.0K Jun 14 01:44 run-llama-1b-meansub
drwxr-xr-x 3 root root 4.0K Jun 13 23:44 run-qwen-0.6b
drwxr-xr-x 3 root root 4.0K Jun 14 00:38 run-qwen-0.6b-meansub
-rw-r--r-- 1 root root 515K Jun 13 22:48 tts_tokenizer.json
-rw-r--r-- 1 root root  11K Jun 13 22:48 tts_tokenizer_stats.json

outputs:
total 1.1G
-rw-r--r-- 1 root root 1.1G Jun 14 00:40 run-qwen-0.6b-artifacts.zip

results:
total 16K
-rw-r--r-- 1 root root 3.4K Jun 14 01:12 run-llama-1b.json
-rw-r--r-- 1 root root 3.4K Jun 14 01:44 run-llama-1b-meansub.json
-rw-r--r-- 1 root root 3.4K Jun 13 23:44 run-qwen-0.6b.json
-rw-r-

## Run aya-base

Model id: `CohereLabs/tiny-aya-base`. This section runs both arms, then immediately displays and archives the full results.

In [16]:
# aya-base / Arm A - random embedding init
!python scripts/model_integration.py --model-id CohereLabs/tiny-aya-base

Building eval dataset: 100%|██████████████| 512/512 [00:00<00:00, 1597.13text/s]
config.json: 1.96kB [00:00, 5.09MB/s]
tokenizer_config.json: 3.26kB [00:00, 9.53MB/s]
tokenizer.json: 100%|██████████████████████| 21.4M/21.4M [00:00<00:00, 25.9MB/s]
Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1874.60text/s]
model.safetensors.index.json: 23.6kB [00:00, 67.4MB/s]
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:26<00:00, 13.10s/it]
Download complete: 100%|████████████████████| 6.70G/6.70G [00:26<00:00, 757MB/s]
Loading weights:   0%|                                  | 0/290 [00:00<?, ?it/s]
Loading weights:   0%| | 1/290 [00:00<00:00, 7002.18it/s, Materializing param=mo
Loading weights:   0%| | 1/290 [00:00<00:00, 2966.27it/s, Materializing param=mo
Loading weights:   1%| | 2/290 [00:03<09:18,  1.94s/it, Materializing param=mode
Loading weights:   1%| | 2/290 [00:03<09:18,  1.94s/it, Materializing param=mode
Loading weights:   1%| | 2/290 [00:03<09:18,  1.9

In [17]:
# aya-base / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id CohereLabs/tiny-aya-base --embedding-init-mode mean_subword

Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1864.04text/s]
Loading weights: 100%|█| 290/290 [00:02<00:00, 129.50it/s, Materializing param=m
Initializing mean-subword embeddings: 100%|█| 8000/8000 [00:02<00:00, 3958.24tok
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '7.234', 'grad_norm': '1.898', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.769', 'grad_norm': '2.718', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.481', 'grad_norm': '0.9213', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.223', 'grad_norm': '0.9674', 'learning_r

## Full results and artifact zip - aya-base

In [18]:
results_aya_base = display_model_results("aya-base")

FULL RESULTS FOR aya-base
Result files:
- random: results/run-aya-base.json
- mean_subword: results/run-aya-base-meansub.json

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
random                          2.9750          1.2586        420720        177986    141420      0.5769
mean_subword                    2.9750          1.2586        420720        177986    141420      0.5769

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is better.
arm                      eval_loss  perplexity    experiment BPB    base BPB   improvement
------------------------------------------------------------------------------------------
random                      4.4127       82.49            1.4755      2.7129        1.2374
mean_subword                3.6593 

In [19]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-aya-base-artifacts.zip     results/run-aya-base.json     results/run-aya-base-meansub.json     models/run-aya-base     models/run-aya-base-meansub
!ls -lh outputs

models:
total 1.8M
-rw-r--r-- 1 root root 522K Jun 13 22:48 asr_tokenizer.json
-rw-r--r-- 1 root root 1.3K Jun 13 22:48 asr_tokenizer_stats.json
-rw-r--r-- 1 root root 515K Jun 13 22:48 mixed_tokenizer.json
-rw-r--r-- 1 root root  12K Jun 13 22:48 mixed_tokenizer_stats.json
-rw-r--r-- 1 root root 227K Jun 13 22:48 router_classifier.pkl
drwxr-xr-x 3 root root 4.0K Jun 14 03:11 run-aya-base
drwxr-xr-x 3 root root 4.0K Jun 14 04:34 run-aya-base-meansub
drwxr-xr-x 3 root root 4.0K Jun 14 01:12 run-llama-1b
drwxr-xr-x 3 root root 4.0K Jun 14 01:44 run-llama-1b-meansub
drwxr-xr-x 3 root root 4.0K Jun 13 23:44 run-qwen-0.6b
drwxr-xr-x 3 root root 4.0K Jun 14 00:38 run-qwen-0.6b-meansub
-rw-r--r-- 1 root root 515K Jun 13 22:48 tts_tokenizer.json
-rw-r--r-- 1 root root  11K Jun 13 22:48 tts_tokenizer_stats.json

outputs:
total 3.0G
-rw-r--r-- 1 root root 1.9G Jun 14 01:48 run-llama-1b-artifacts.zip
-rw-r--r-- 1 root root 1.1G Jun 14 00:40 run-qwen-0.6b-artifacts.zip

results:
total 24K
-rw-r--r

## Notebook-level summary

In [20]:
MODEL_SLUGS = ['qwen-0.6b', 'llama-1b', 'aya-base']
all_results = {slug: {arm: load_result(path) for arm, path in result_paths(slug).items()} for slug in MODEL_SLUGS}

print("=" * 88)
print("NOTEBOOK SPLIT SUMMARY")
print("=" * 88)
print_tokenization_table({f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()})
print_bpb_table({f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()})

summary = {
    slug: {
        arm: {
            "eval_loss": payload["eval"]["eval_loss"],
            "perplexity": payload["eval"]["perplexity"],
            "experiment_bpb": payload["eval"]["bpb"]["experiment"]["bits_per_byte"],
            "base_bpb": (payload["eval"]["bpb"].get("base") or {}).get("bits_per_byte"),
            "bpb_improvement": payload["eval"]["bpb"].get("improvement"),
            "base_fertility": payload["token_count_comparison"]["base_model_tokenizer"]["fertility"],
            "akan_fertility": payload["token_count_comparison"]["experiment_tokenizer"]["fertility"],
            "token_reduction_ratio": payload["token_count_comparison"]["token_reduction_ratio"],
        }
        for arm, payload in arms.items()
    }
    for slug, arms in all_results.items()
}
Path("outputs").mkdir(exist_ok=True)
Path("outputs/light-split-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote outputs/light-split-summary.json")


NOTEBOOK SPLIT SUMMARY

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
qwen-0.6b/random                2.5303          1.2586        357830        177986    141420      0.5026
qwen-0.6b/mean_subword          2.5303          1.2586        357830        177986    141420      0.5026
llama-1b/random                 3.0732          1.2586        434611        177986    141420      0.5905
llama-1b/mean_subword           3.0732          1.2586        434611        177986    141420      0.5905
aya-base/random                 2.9750          1.2586        420720        177986    141420      0.5769
aya-base/mean_subword           2.9750          1.2586        420720        177986    141420      0.5769

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is

In [21]:
!ls -lh outputs

total 5.4G
-rw-r--r-- 1 root root 2.3K Jun 14 04:38 light-split-summary.json
-rw-r--r-- 1 root root 2.4G Jun 14 04:38 run-aya-base-artifacts.zip
-rw-r--r-- 1 root root 1.9G Jun 14 01:48 run-llama-1b-artifacts.zip
-rw-r--r-- 1 root root 1.1G Jun 14 00:40 run-qwen-0.6b-artifacts.zip
